# Teencode normalizer — Colab GPU training
1. **Runtime > Change runtime type > GPU** (T4 is fine; L4/A100 faster), save.
2. Prepare your data locally (`src/train/encode_data.py` on the HF dataset, or `src/train/step_1_prepare_data.py` on raw pairs; see the README), zip the resulting folder as `data.zip` (must contain `train.jsonl`, `dev*.jsonl`, `tokenizer.json`, `output_vocab.txt`) and upload it to **MyDrive/teencode_colab/** (or upload when asked). Optionally upload an init checkpoint as `MyDrive/teencode_colab/init.pt`.
3. Set `REPO_URL` in the setup cell to your fork, then **Runtime > Run all**.
4. If Colab disconnects: reconnect and **Run all** again — it resumes from `last.pt` in Drive.
5. The last cell downloads a results zip (checkpoints, logs, INT8 ONNX).

In [ ]:
!nvidia-smi || echo 'No GPU! Runtime > Change runtime type > GPU'

In [ ]:
USE_DRIVE = True   # mount your own Google Drive (recommended: checkpoints survive disconnects)
import os
DRIVE_DIR = '/content/drive/MyDrive/teencode_colab'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); os.makedirs(DRIVE_DIR, exist_ok=True)
    except Exception as e:
        print('Drive mount failed, falling back to /content:', e); USE_DRIVE = False
BASE = DRIVE_DIR if USE_DRIVE else '/content/teencode_colab'; os.makedirs(BASE, exist_ok=True)
print('Runs will be saved under', BASE)

In [ ]:
import os, shutil, subprocess
REPO_URL = 'https://github.com/<you>/teencode-normalizer'   # <- change to your repo
WORK = '/content/teencode-normalizer'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, WORK], check=True)
os.chdir(WORK)
!pip -q install -r src/train/requirements.txt
dz = os.path.join(BASE, 'data.zip')
if not os.path.exists(dz) and os.path.exists('/content/data.zip'): shutil.copy('/content/data.zip', dz)
if not os.path.exists(dz):
    print('data.zip not found in', BASE, '- please upload it now')
    from google.colab import files
    up = files.upload(); shutil.move([n for n in up if n.endswith('.zip')][0], dz)
if not os.path.exists('data/train.jsonl'):
    os.makedirs('data_tmp', exist_ok=True); subprocess.run(['unzip', '-oq', dz, '-d', 'data_tmp'], check=True)
    root = next(r for r, _, fs in os.walk('data_tmp') if 'train.jsonl' in fs)
    shutil.rmtree('data', ignore_errors=True); shutil.move(root, 'data')
print(sorted(os.listdir('data')))
INIT_CKPT = os.path.join(BASE, 'init.pt') if os.path.exists(os.path.join(BASE, 'init.pt')) else ''
print('init checkpoint:', INIT_CKPT or '(none, training from scratch)')

In [ ]:
# ---- config ----
RUN_NAME   = 'gpu_H'
ENC_LAYERS, DEC_LAYERS, D_FF, D_MODEL = 8, 4, 1536, 256   # config H (~19.9M params)
EPOCHS     = 5
MAX_TOKENS = 24000     # tokens per batch (CPU run used 6000). 4x batch -> LR ~2x
BATCH_SIZE = 0         # optional cap on sentences per batch, 0 = none
LR         = 6e-4
WARMUP     = 500
EVAL_EVERY = 500
SAVE_EVERY = 200       # last.pt to Drive every N steps
DEV_SETS   = ','.join(f'data/{f}' for f in sorted(os.listdir('data')) if f.startswith('dev') and f.endswith('.jsonl'))   # or e.g. 'data/dev.jsonl:0.4,data/dev_untoned.jsonl:0.3,...'
RUN_DIR = f'{BASE}/runs/{RUN_NAME}'; os.makedirs(RUN_DIR, exist_ok=True)
print(RUN_DIR)

In [ ]:
import subprocess, sys
resume = os.path.exists(f'{RUN_DIR}/last.pt')
cmd = [sys.executable, '-u', 'src/train/step_2_train.py', '--data', 'data', '--ckpt-dir', RUN_DIR, '--device', 'auto',
       '--epochs', str(EPOCHS), '--max-tokens', str(MAX_TOKENS), '--batch-size', str(BATCH_SIZE), '--lr', str(LR),
       '--warmup', str(WARMUP), '--sched', 'cosine', '--eval-every', str(EVAL_EVERY), '--save-every', str(SAVE_EVERY),
       '--num-workers', '2', '--threads', '2', '--dev-sets', DEV_SETS, '--backup-dir', '', '--log', '',
       '--enc-layers', str(ENC_LAYERS), '--dec-layers', str(DEC_LAYERS), '--d-ff', str(D_FF), '--d-model', str(D_MODEL)]
if resume: cmd += ['--resume']; print('Resuming from', f'{RUN_DIR}/last.pt')
elif INIT_CKPT: cmd += ['--init-from', INIT_CKPT, '--init-map', 'identity', '--freeze-init-steps', '500']
log = open(f'{RUN_DIR}/train.log', 'a'); log.write('[cmd] ' + ' '.join(cmd) + '\n'); log.flush()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end=''); log.write(line); log.flush()
p.wait(); log.close(); print('exit code', p.returncode)

In [ ]:
# optional: INT8 ONNX export + quick greedy dev eval
RUN_EXPORT = True
if RUN_EXPORT:
    !python src/train/step_3_export_onnx.py --ckpt "{RUN_DIR}/best.pt" --data data --out "{RUN_DIR}/onnx_int8" --verify 20 2>&1 | tee -a "{RUN_DIR}/export.log"
    !python src/train/colab_eval.py --ckpt "{RUN_DIR}/best.pt" --data data --n 1000 2>&1 | tee "{RUN_DIR}/dev_eval.log" 

In [ ]:
# zip results and download (send this zip back in chat)
import zipfile
from google.colab import files
out = f'/content/{RUN_NAME}_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('best.pt', 'last.pt', 'train.log', 'export.log', 'dev_eval.log'):
        if os.path.exists(f'{RUN_DIR}/{f}'): z.write(f'{RUN_DIR}/{f}', f)
    od = f'{RUN_DIR}/onnx_int8'
    if os.path.isdir(od):
        for f in os.listdir(od):
            if f.endswith('.int8.onnx') or f.endswith('.json'): z.write(f'{od}/{f}', f'onnx_int8/{f}')
print(out, os.path.getsize(out) / 1e6, 'MB'); files.download(out)